# Week 97, Day 4: Code review interview

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w097d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('orders.py', 'w', encoding='utf-8') as f:
    f.write('import json\n\n\ndef average_basket(orders):\n    return sum(o["total"] for o in orders) / len(orders)\n\n\ndef add_tag(order, tag, tags=[]):\n    tags.append(tag)\n    order["tags"] = tags\n    return order\n\n\ndef load_orders(path):\n    try:\n        f = open(path)\n        return json.load(f)\n    except:\n        return []\n\n\ndef find_customer(conn, name):\n    return conn.execute(f"SELECT * FROM customer WHERE name = \'{name}\'").fetchall()\n\n\ndef apply_discount(prices, percent):\n    for i in range(len(prices)):\n        prices[i] = round(prices[i] * (100 - percent) / 100, 2)\n    return prices\n\n\ndef unique_customers(orders):\n    seen = []\n    for o in orders:\n        if o["customer"] not in seen:\n            seen.append(o["customer"])\n    return seen\n')

**Predict before you run:** what will this print?



In [ ]:
from orders import add_tag

print(add_tag({}, "gift"), add_tag({}, "urgent"))

> **Key idea.** Ask what the code is for, read it all once, then report the most severe problems first: security, correctness and data loss, crashes, performance, style. One sentence each: what's wrong, its effect, the fix, and the test that would catch it. Be kind and specific.

### Exercise: Most severe first

Write `prioritise(findings)`: each finding is `(name, category)` with categories `security`, `correctness`, `crash`, `performance` and `style`. Return the names, most severe first, in the order the categories are listed here; keep the original order within a category.

In [ ]:
def prioritise(findings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rank():\n    \"say these first\"\n    findings = [(\"unique_customers is quadratic\", \"performance\"), (\"add_tag shares a list\", \"correctness\"),\n                (\"find_customer builds SQL\", \"security\"), (\"average_basket fails on []\", \"crash\"),\n                (\"load_orders hides errors\", \"correctness\"), (\"short names\", \"style\")]\n    assert prioritise(findings) == [\"find_customer builds SQL\", \"add_tag shares a list\", \"load_orders hides errors\",\n                                    \"average_basket fails on []\", \"unique_customers is quadratic\", \"short names\"]")

### Exercise: Errors that disappeared

`load_orders` leaves the file open and its bare `except:` turns every problem (a typo in the path, broken JSON, even Ctrl+C) into an empty list. Rewrite it: open the file with `with`, return `[]` only if the file doesn't exist, and let a `json.JSONDecodeError` (a `ValueError`) propagate.

In [ ]:
import json

def load_orders(path):
    try:
        f = open(path)
        return json.load(f)
    except:
        return []

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json\n\ndef test_load():\n    \"only a missing file means no orders\"\n    with open(\"orders.json\", \"w\") as f:\n        json.dump([{\"customer\": \"ada\", \"total\": 12.5}], f)\n    assert load_orders(\"orders.json\") == [{\"customer\": \"ada\", \"total\": 12.5}]\n    assert load_orders(\"no-such-file.json\") == []\n    with open(\"broken.json\", \"w\") as f:\n        f.write(\"[{\")\n    try:\n        load_orders(\"broken.json\")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"broken JSON should raise\")\n    assert \"with open\" in source, \"close the file with a with statement\"")

### Exercise: SQL from a string

`find_customer` puts the name straight into the SQL, so `x' OR '1'='1` returns every customer. Rewrite it to use a parameter: `conn.execute("SELECT * FROM customer WHERE name = ?", (name,))`, then `.fetchall()`.

In [ ]:
def find_customer(conn, name):
    return conn.execute(f"SELECT * FROM customer WHERE name = '{name}'").fetchall()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class FakeConnection:\n    def __init__(self):\n        self.calls = []\n    def execute(self, sql, params=()):\n        self.calls.append((sql, params))\n        return self\n    def fetchall(self):\n        return [(\"ada\",)]\n\ndef test_parameter():\n    \"the name never enters the SQL\"\n    conn = FakeConnection()\n    assert find_customer(conn, \"x' OR '1'='1\") == [(\"ada\",)]\n    sql, params = conn.calls[0]\n    assert \"x'\" not in sql and \"?\" in sql and params == (\"x' OR '1'='1\",)")

### Exercise: A side effect nobody asked for

`apply_discount` changes the caller's list as well as returning it, so after a "what if 10% off?" calculation the original prices are gone. Rewrite it to return a **new** list and leave `prices` unchanged.

In [ ]:
def apply_discount(prices, percent):
    for i in range(len(prices)):
        prices[i] = round(prices[i] * (100 - percent) / 100, 2)
    return prices

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_discount():\n    \"the input is left alone\"\n    prices = [10.0, 20.0, 5.5]\n    assert apply_discount(prices, 10) == [9.0, 18.0, 4.95]\n    assert prices == [10.0, 20.0, 5.5]\n    assert apply_discount([], 50) == []")

### Exercise: Quadratic without meaning to be

`unique_customers` checks `in` on a list, which scans it each time: fine for ten orders, slow for a hundred thousand. Rewrite it in linear time, keeping the customers in order of first appearance.

In [ ]:
def unique_customers(orders):
    seen = []
    for o in orders:
        if o["customer"] not in seen:
            seen.append(o["customer"])
    return seen

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import time\n\ndef test_unique():\n    \"first appearance order\"\n    orders = [{\"customer\": c} for c in [\"ada\", \"bo\", \"ada\", \"cy\", \"bo\"]]\n    assert unique_customers(orders) == [\"ada\", \"bo\", \"cy\"] and unique_customers([]) == []\n\ndef test_fast():\n    \"100,000 orders, 50,000 customers\"\n    orders = [{\"customer\": f\"c{i % 50_000}\"} for i in range(100_000)]\n    start = time.perf_counter()\n    assert len(unique_customers(orders)) == 50_000\n    assert time.perf_counter() - start < 1.5, \"too slow: a set for membership\"")

### Bug hunt: An average of nothing

`average_basket([])` divides by zero. Return `0.0` for no orders, so this prints `0.0 15.0`.

In [ ]:
def average_basket(orders):
    return sum(o["total"] for o in orders) / len(orders)

print(average_basket([]), average_basket([{"total": 10}, {"total": 20}]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"empty input is ordinary input\"\n    expect_output(\"0.0 15.0\")")

### Bug hunt: Tags that leaked between orders

The default list is shared by every call, so the second order gets both tags and this prints `{'tags': ['gift', 'urgent']} {'tags': ['gift', 'urgent']}`. Use `None` as the default and make a new list, so it prints `{'tags': ['gift']} {'tags': ['urgent']}`.

In [ ]:
def add_tag(order, tag, tags=[]):
    tags.append(tag)
    order["tags"] = tags
    return order

print(add_tag({}, "gift"), add_tag({}, "urgent"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a new list each call\"\n    expect_output(\"{'tags': ['gift']} {'tags': ['urgent']}\")")

### Puzzle: Loading safely

Arrange the corrected `load_orders`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the most severe category of problem in orders.py
q = ___   # the placeholder sqlite3 uses for a parameter
r = ___   # the time complexity of `x in some_list`, as "O(...)"
s = ___   # True or False: a bare except: also catches KeyboardInterrupt
t = ___   # how many problems orders.py has, as described today

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"security\" and q == \"?\" and r.replace(\" \", \"\") == \"O(n)\", \"p, q and r\"\n    assert s is True and t == 6, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the module, corrected

Write the whole of `orders.py` again with all six problems fixed: `average_basket` (0.0 for no orders), `add_tag` (no shared default), `load_orders` (`with`, only a missing file gives `[]`), `find_customer` (a parameter), `apply_discount` (a new list) and `unique_customers` (linear, in first-appearance order).

In [ ]:
import json

def average_basket(orders):
    ___

def add_tag(order, tag, tags=None):
    ___

def load_orders(path):
    ___

def find_customer(conn, name):
    ___

def apply_discount(prices, percent):
    ___

def unique_customers(orders):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import json, time\n\nclass FakeConnection:\n    def __init__(self):\n        self.calls = []\n    def execute(self, sql, params=()):\n        self.calls.append((sql, params))\n        return self\n    def fetchall(self):\n        return []\n\ndef test_all():\n    \"six fixes\"\n    assert average_basket([]) == 0.0 and average_basket([{\"total\": 3}, {\"total\": 5}]) == 4.0\n    assert add_tag({}, \"gift\") == {\"tags\": [\"gift\"]} and add_tag({}, \"urgent\") == {\"tags\": [\"urgent\"]}\n    assert load_orders(\"still-no-such-file.json\") == []\n    with open(\"bad.json\", \"w\") as f:\n        f.write(\"{\")\n    try:\n        load_orders(\"bad.json\")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"broken JSON\")\n    conn = FakeConnection()\n    find_customer(conn, \"o'brien\")\n    assert conn.calls == [(\"SELECT * FROM customer WHERE name = ?\", (\"o'brien\",))]\n    prices = [10.0, 4.0]\n    assert apply_discount(prices, 25) == [7.5, 3.0] and prices == [10.0, 4.0]\n    orders = [{\"customer\": f\"c{i % 50_000}\"} for i in range(100_000)]\n    start = time.perf_counter()\n    assert unique_customers(orders)[:2] == [\"c0\", \"c1\"]\n    assert time.perf_counter() - start < 1.5")

## Warm-up

Two unrelated orders now share one list of tags. That's one of the six problems in `orders.py`; read the whole module before going on, and write down every problem you can find in five minutes.

## How the interview goes

1. **Ask what it's for** and how it's used: who calls `find_customer`, and with what input? The answers change what matters.
2. **Read all of it once** before commenting, so your first remark isn't about a typo next to a security hole.
3. **Say the important things first**, each in one sentence: what's wrong, what goes wrong because of it, and the fix. "`find_customer` builds SQL from the name, so a name like `x' OR '1'='1` returns every customer; use a parameter: `execute('... WHERE name = ?', (name,))`."
4. **Mention tests**: which test would have caught it.
5. **Be kind and concrete**, as in Week 96: describe the code, not its author, and acknowledge what's good.

## Severity

A simple order: **security** (injection, secrets, unsafe deserialisation), then **correctness and data loss** (wrong answers, shared state, swallowed errors), then **crashes** on ordinary input (empty lists, missing files), then **performance** on realistic sizes (quadratic loops), then **clarity and style**. An interviewer who hears you rank problems this way knows you'd review their pull requests usefully.

## Practice

## Quiz

## Daily challenge

## A mock round

Ask a friend to pick a short module from a project you don't know (or take one from Week 79's vulnerable notes service) and give you twenty minutes. Say your findings aloud, most severe first, one sentence each, then compare with what they found. Record yourself once: most people discover they start with style.

## Before you go

Tomorrow is the behavioural interview: the questions about how you work, answered with real stories from this course.